# 06 · K15/GRU Soft Selector — 이질 backbone per-sample 결합 (EXP #32, LB 0.6910)

직전 = K15(LGBM Frenet, LB 0.6906). 단일 backbone의 feature/weight 튜닝 라인은 saturate, ensemble(blend)은 양방향 closed.

**가정**: blend는 실패했지만 **per-sample selector는 별개 메커니즘**이다. K15(tree)와 GRU(NN)는 이질 backbone이라 oracle ceiling이 +0.0221(minor +0.0533, K10/K15의 2배, #31 정량). sample마다 더 맞는 모델을 고르면(soft blend) 약한 classifier로도 이 ceiling의 일부를 회수한다.

**실험**: K15·GRU·classifier cache 재사용(학습 0~). recovery-area(K15_only ∪ GRU_only 432 sample) AUC를 primary sanity로, 6 mode(Hard/Soft/Threshold/Minor-rule 등)를 HR 4th guard + floor로 평가.

**결과**: **Soft(raw p_clf 가중) 단독 5-gate 통과 → LB 0.6910 (+0.0004)**. post-hoc transfer fail 시리즈(#22/#28/#31)를 처음 깬 양수 transfer — "이질 backbone per-sample blend"라는 새 lever type을 연다. 회수율은 ceiling의 17%, 미회수 83%가 다음 lever 잠재 영역.

## 셀 0 — imports + 상수

In [ ]:
import os, time, json
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score

DT, DT_PRED = 0.04, 0.08
N_FOLDS = 5
MINORITY_THRESHOLD = 5.0
HIT_RADIUS = 0.01
SEEDS = [42, 7, 123]

# Classifier hyperparam (LGBM binary, #31 동일)
clf_params = dict(
    objective='binary', metric='binary_logloss',
    learning_rate=0.05, num_leaves=31, min_data_in_leaf=10,
    max_bin=511, n_estimators=300, random_state=42, verbose=-1,
)

# K15/GRU reference HR (EXP #29 cache + EXP #25 cache, #31 셀 5.5 정량)
K15_REF_HR = dict(overall=0.6649, major=0.7312, minor=0.3105)
GRU_REF_OVERALL = 0.6659  # EXP #25 V2 Frenet GRU solo OOF (3-seed mean)
GRU_REF_MINOR = 0.3181

# K15/GRU 2-way oracle ceiling (#31 셀 5.5)
ORACLE_KG_HR = dict(overall=0.6870, major=0.7474, minor=0.3638)
CEILING_GAIN_KG = dict(overall=0.0221, major=0.0162, minor=0.0533)

# Floor baseline (#31 minor→GRU rule 학습 0 정량)
FLOOR_DO = 0.0012   # minor→GRU rule Δo (학습 0)

# 임계 (재설계 2026-05-28 후반, #31 lesson)
# Phase 1/2a/2b sanity gates
AUC_RECOVERY_THR = 0.55     # primary sanity: K15_only ∪ GRU_only 직접 AUC
AUC_OVERALL_SANITY = 0.50   # informational: negative learning 차단
FOLD_STD_MAX = 0.020        # 학습 안정성

# Phase 3 HR primary gate (vs K15 OOF 0.6649)
COMBINED_STD = 0.0014
G1_THR = COMBINED_STD * (6 ** 0.5)   # ≈ 0.0034 (Bonferroni √6 for 6 mode multi-comparison)
G4_THR = COMBINED_STD * (2 ** 0.5)   # ≈ 0.0020 (per-mode LB submission)

os.makedirs('results', exist_ok=True)
print(f'SEEDS={SEEDS}')
print(f'AUC_RECOVERY_THR={AUC_RECOVERY_THR} (primary sanity), AUC_OVERALL_SANITY={AUC_OVERALL_SANITY}')
print(f'G1_THR={G1_THR:.4f} (multi-comparison √6), G4_THR={G4_THR:.4f} (per-mode), FLOOR_DO={FLOOR_DO:.4f}')
print(f'K15 ref HR overall={K15_REF_HR["overall"]:.4f}, K15/GRU ceiling +{CEILING_GAIN_KG["overall"]:.4f}')


## 셀 1 — helper

In [ ]:
def _norm(x):
    return np.linalg.norm(x, axis=-1)


def physics_baseline(traj):
    p0, p_m40 = traj[:, -1, :], traj[:, -2, :]
    v_last = (p0 - p_m40) / DT
    return (p0 + v_last * DT_PRED).astype(np.float32)


def hit_rate(pred, label, r=HIT_RADIUS):
    return float((np.linalg.norm(pred - label, axis=1) < r).mean())


def make_splits(stratify_int, seed, n_folds=N_FOLDS):
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    return [(tr, va) for tr, va in skf.split(np.arange(len(stratify_int)), stratify_int)]


def hr_triple(hit_arr, minority_mask):
    return dict(
        overall=float(hit_arr.mean()),
        major=float(hit_arr[~minority_mask].mean()),
        minor=float(hit_arr[minority_mask].mean()),
    )


def run_classifier_seeds(X_tr, y_target, X_ts, stratify, seeds=SEEDS, restrict_idx=None):
    """Train classifier with 3 seed × 5 fold = 15 model.

    Args:
        X_tr, y_target: training features and binary target (length N or restricted)
        X_ts: test features (None for skip)
        stratify: array for StratifiedKFold (same length as y_target)
        restrict_idx: if given, only use this subset for training (for Phase 2b)

    Returns:
        dict with oof_prob (length N, NaN where not predicted), test_prob, per_seed metadata
    """
    N = len(y_target)
    oof_probs = []
    test_probs = [] if X_ts is not None else None
    per_seed = []

    for seed in seeds:
        folds = make_splits(stratify, seed=seed)
        oof_prob = np.full(N, np.nan, dtype=np.float32)
        test_prob_seed = np.zeros(len(X_ts), dtype=np.float32) if X_ts is not None else None
        fold_accs = []
        t0 = time.time()

        for tr_idx, val_idx in folds:
            m = lgb.LGBMClassifier(**clf_params)
            m.fit(X_tr[tr_idx], y_target[tr_idx],
                  eval_set=[(X_tr[val_idx], y_target[val_idx])],
                  callbacks=[lgb.early_stopping(30, verbose=False)])
            p_val = m.predict_proba(X_tr[val_idx])[:, 1].astype(np.float32)
            oof_prob[val_idx] = p_val
            if X_ts is not None:
                test_prob_seed += m.predict_proba(X_ts)[:, 1].astype(np.float32) / N_FOLDS
            fold_accs.append(accuracy_score(y_target[val_idx], (p_val > 0.5).astype(int)))

        oof_probs.append(oof_prob)
        if X_ts is not None:
            test_probs.append(test_prob_seed)
        acc_seed = accuracy_score(y_target, (oof_prob > 0.5).astype(int))
        auc_seed = roc_auc_score(y_target, oof_prob)
        fold_std = float(np.std(fold_accs, ddof=1))
        per_seed.append(dict(seed=seed, accuracy=float(acc_seed), auc=float(auc_seed),
                              fold_accs=[float(a) for a in fold_accs],
                              fold_std=fold_std, time_sec=time.time() - t0))

    oof_prob_mean = np.mean(oof_probs, axis=0).astype(np.float32)
    test_prob_mean = np.mean(test_probs, axis=0).astype(np.float32) if X_ts is not None else None
    return dict(oof_prob_mean=oof_prob_mean, test_prob_mean=test_prob_mean,
                per_seed=per_seed, oof_probs_per_seed=oof_probs,
                test_probs_per_seed=test_probs)


def compute_classifier_metrics(oof_prob_mean, y_target, recovery_mask=None, per_seed=None):
    """Compute overall acc/AUC + recovery-area AUC + fold std."""
    overall_acc = float(accuracy_score(y_target, (oof_prob_mean > 0.5).astype(int)))
    overall_auc = float(roc_auc_score(y_target, oof_prob_mean))
    if recovery_mask is not None and recovery_mask.sum() > 0:
        rec_auc = float(roc_auc_score(y_target[recovery_mask], oof_prob_mean[recovery_mask]))
    else:
        rec_auc = float('nan')
    fold_std_mean = float(np.mean([m['fold_std'] for m in per_seed])) if per_seed else float('nan')
    return dict(overall_acc=overall_acc, overall_auc=overall_auc,
                recovery_auc=rec_auc, fold_std_mean=fold_std_mean)


print('helper 정의 완료 (run_classifier_seeds + compute_classifier_metrics 포함)')


## 셀 2 — Drive mount + 데이터 + K15/GRU cache 동기화

K15 cache (EXP #29) + GRU cache (EXP #25) **둘 다 필수**. GRU cache 누락 시 EXP #32 진행 불가.


In [ ]:
CACHE_TRAJ_TR = 'traj_train.npy'
CACHE_Y_TR    = 'y_train.npy'
CACHE_TRAJ_TS = 'traj_test.npy'

K15_OOF_PATHS  = [f'results/topk_e29_oof_K15_seed{s}.npy' for s in SEEDS]
K15_TEST_PATHS = [f'results/topk_e29_test_K15_seed{s}.npy' for s in SEEDS]
GRU_OOF_PATHS  = [f'results/gru_e25_fren_oof_seed{s}.npy' for s in SEEDS]
GRU_TEST_PATHS = [f'results/gru_e25_fren_test_seed{s}.npy' for s in SEEDS]
all_required = K15_OOF_PATHS + K15_TEST_PATHS + GRU_OOF_PATHS + GRU_TEST_PATHS

need_mount = (
    not (os.path.exists(CACHE_TRAJ_TR) and os.path.exists(CACHE_Y_TR))
    or any(not os.path.exists(p) for p in all_required)
)

if need_mount:
    from google.colab import drive
    drive.mount('/content/drive')
    ZIP_PATH = '/content/drive/MyDrive/open.zip'
    if not os.path.exists('/content/open'):
        !unzip -q -o "{ZIP_PATH}" -d /content/

# K15 + GRU cache Drive→로컬 동기화
for p in all_required:
    if not os.path.exists(p):
        drive_p = f'/content/drive/MyDrive/{p}'
        if os.path.exists(drive_p):
            !cp {drive_p} {p}
missing = [p for p in all_required if not os.path.exists(p)]
assert not missing, f'K15/GRU cache 누락 (필수): {missing}'
print(f'K15 + GRU cache 12/12 OK')

def _resolve_data_dir():
    for cand in ['/content/open', '/content']:
        if os.path.exists(f'{cand}/train_labels.csv'):
            return cand
    return None

DATA_DIR = _resolve_data_dir()

def _resolve_sample_sub(data_dir):
    for p in [f'{data_dir}/sample_submission.csv', f'{data_dir}/submission/sample_submission.csv']:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f'sample_submission.csv not found under {data_dir}')

if os.path.exists(CACHE_TRAJ_TR) and os.path.exists(CACHE_Y_TR):
    traj_train = np.load(CACHE_TRAJ_TR)
    y_train    = np.load(CACHE_Y_TR)
else:
    labels = pd.read_csv(f'{DATA_DIR}/train_labels.csv')
    train_ids = labels['id'].tolist()
    y_train = labels[['x','y','z']].values.astype(np.float32)
    traj_train = np.empty((len(train_ids), 11, 3), dtype=np.float32)
    for i, tid in enumerate(train_ids):
        df = pd.read_csv(f'{DATA_DIR}/train/{tid}.csv')
        traj_train[i] = df[['x','y','z']].values
    np.save(CACHE_TRAJ_TR, traj_train)
    np.save(CACHE_Y_TR, y_train)

sample_sub_path = _resolve_sample_sub(DATA_DIR)
sample_sub = pd.read_csv(sample_sub_path)
test_ids = sample_sub['id'].tolist()
if os.path.exists(CACHE_TRAJ_TS):
    traj_test = np.load(CACHE_TRAJ_TS)
else:
    traj_test = np.empty((len(test_ids), 11, 3), dtype=np.float32)
    for i, tid in enumerate(test_ids):
        df = pd.read_csv(f'{DATA_DIR}/test/{tid}.csv')
        traj_test[i] = df[['x','y','z']].values
    np.save(CACHE_TRAJ_TS, traj_test)

assert traj_train.shape == (10000, 11, 3) and traj_test.shape == (10000, 11, 3)
print(f'train {traj_train.shape}, test {traj_test.shape}, cache 12/12 OK')


## 셀 3 — base + Frenet frame + minority mask

In [ ]:
def build_frenet_batch(v_last, a_sm, fb_thresh=1e-6):
    vn = np.linalg.norm(v_last, axis=1, keepdims=True)
    eL = v_last / (vn + 1e-9)
    a_perp = a_sm - (a_sm * eL).sum(1, keepdims=True) * eL
    apn = np.linalg.norm(a_perp, axis=1, keepdims=True)
    eN_p = a_perp / (apn + 1e-9)
    z_up = np.array([0., 0., 1.]); y_up = np.array([0., 1., 0.])
    n1 = np.linalg.norm(np.cross(eL, z_up), axis=1, keepdims=True)
    eN_fb1 = np.cross(eL, z_up) / (n1 + 1e-9)
    eN_fb2 = np.cross(eL, y_up); n2 = np.linalg.norm(eN_fb2, axis=1, keepdims=True)
    eN_fb  = np.where(n1 < 1e-6, eN_fb2 / (n2 + 1e-9), eN_fb1)
    eN = np.where(apn < fb_thresh, eN_fb, eN_p)
    eB = np.cross(eL, eN)
    eB = eB / (np.linalg.norm(eB, axis=1, keepdims=True) + 1e-9)
    return eL.astype(np.float32), eN.astype(np.float32), eB.astype(np.float32)


def compute_kinematics(traj):
    v = (traj[:, 1:, :] - traj[:, :-1, :]) / DT
    a = (v[:, 1:, :] - v[:, :-1, :]) / DT
    v_last = v[:, -1, :]
    a_last = a[:, -1, :]
    jerk_last = (a[:, -1, :] - a[:, -2, :]) / DT
    w = np.array([1, 2, 3]) / 6
    a_sm = (a[:, -3:, :] * w[None, :, None]).sum(axis=1)
    return v_last.astype(np.float32), a_last.astype(np.float32), jerk_last.astype(np.float32), a_sm.astype(np.float32)


base_train = physics_baseline(traj_train)
base_test  = physics_baseline(traj_test)
vl_tr, al_tr, jl_tr, asm_tr = compute_kinematics(traj_train)
vl_ts, al_ts, jl_ts, asm_ts = compute_kinematics(traj_test)
eL_tr, eN_tr, eB_tr = build_frenet_batch(vl_tr, asm_tr)
eL_ts, eN_ts, eB_ts = build_frenet_batch(vl_ts, asm_ts)

acc_norm_last_tr = _norm(al_tr)
acc_norm_last_ts = _norm(al_ts)
minority_mask_tr = acc_norm_last_tr >= MINORITY_THRESHOLD
minority_mask_ts = acc_norm_last_ts >= MINORITY_THRESHOLD
print(f'minority train {minority_mask_tr.sum()}/{len(minority_mask_tr)} ({minority_mask_tr.mean():.4f})')
print(f'minority test  {minority_mask_ts.sum()}/{len(minority_mask_ts)} ({minority_mask_ts.mean():.4f})')

# Sanity: train/test minority 분포 차이 ±5% 이내 (#32 위험 mitigation)
mr_diff = abs(minority_mask_tr.mean() - minority_mask_ts.mean())
if mr_diff > 0.05:
    print(f'⚠ train/test minority 분포 차이 {mr_diff:.4f} > 0.05 — minor→GRU lever 결과 해석 주의')
else:
    print(f'  train/test minority 분포 정렬 OK (Δ {mr_diff:.4f} ≤ 0.05)')


## 셀 4 — V3 15 (Phase 1) + V3 26 (Phase 2a) feature builders

V3 15 = EXP #29 채택 baseline. V3 26 = EXP #29 simplicity 제거 11 feat 복원.


In [ ]:
FEAT_NAMES_V3_15 = [
    'a_tang_last', 'va_dot', 'speed_last', 'j_L', 'cos_turn_last',
    'j_N', 'a_cent_last', 'speed_diff_half', 'turn_mean_half_diff', 'acc_norm_last',
    'radial_v', 'turn_mean', 'acc_norm_w3', 'distance_r', 'vy_std',
]
assert len(FEAT_NAMES_V3_15) == 15


def build_v3_15(traj, eL, eN, eB):
    p = traj
    v = (p[:, 1:, :] - p[:, :-1, :]) / DT
    a = (v[:, 1:, :] - v[:, :-1, :]) / DT
    v_last, a_last = v[:, -1, :], a[:, -1, :]
    speed_last, acc_norm_last = _norm(v_last), _norm(a_last)
    w_sm = np.array([1, 2, 3]) / 6
    a3 = a[:, -3:, :]
    a_w3 = (a3 * w_sm[None, :, None]).sum(axis=1)
    acc_norm_w3 = _norm(a_w3)
    p0 = p[:, -1, :]
    distance_r = _norm(p0)
    radial_v = (p0 * v_last).sum(1) / (distance_r + 1e-9)
    v_n = v / (_norm(v)[..., None] + 1e-9)
    cos_consec = (v_n[:, 1:, :] * v_n[:, :-1, :]).sum(-1).clip(-1, 1)
    turn = np.arccos(cos_consec)
    turn_mean, cos_turn_last = turn.mean(1), cos_consec[:, -1]
    va_dot = (v_last * a_last).sum(1) / (speed_last * acc_norm_last + 1e-9)
    jerk_last = (a[:, -1, :] - a[:, -2, :]) / DT
    a_tang_last = (v_last * a_last).sum(1) / (speed_last + 1e-9)
    a_cent_last = _norm(np.cross(v_last, a_last)) / (speed_last + 1e-9)
    speed_seq = _norm(v)
    speed_diff_half = speed_seq[:, 5:].mean(1) - speed_seq[:, :5].mean(1)
    turn_mean_half_diff = turn[:, 4:].mean(1) - turn[:, :4].mean(1)
    vy_std = v[:, :, 1].std(axis=1)
    j_L = (jerk_last * eL).sum(1)
    j_N = (jerk_last * eN).sum(1)
    X = np.column_stack([
        a_tang_last, va_dot, speed_last, j_L, cos_turn_last,
        j_N, a_cent_last, speed_diff_half, turn_mean_half_diff, acc_norm_last,
        radial_v, turn_mean, acc_norm_w3, distance_r, vy_std,
    ]).astype(np.float32)
    return X


FEAT_NAMES_V3_26 = [
    'speed_last','acc_norm_last','acc_norm_w3',
    'vx_std','vy_std','vz_std','ax_std','ay_std','az_std',
    'path_eff','distance_r','radial_v',
    'turn_mean','cos_turn_last','va_dot',
    'a_tang_last','a_cent_last','speed_diff_half','turn_mean_half_diff',
    'a_N','a_B','j_L','j_N','j_B','aw_L','aw_N',
]
assert len(FEAT_NAMES_V3_26) == 26


def build_v3_26(traj, eL, eN, eB):
    p = traj
    v = (p[:, 1:, :] - p[:, :-1, :]) / DT
    a = (v[:, 1:, :] - v[:, :-1, :]) / DT
    v_last, a_last = v[:, -1, :], a[:, -1, :]
    speed_last, acc_norm_last = _norm(v_last), _norm(a_last)
    w_sm = np.array([1, 2, 3]) / 6
    a3 = a[:, -3:, :]
    a_w3 = (a3 * w_sm[None, :, None]).sum(axis=1)
    acc_norm_w3 = _norm(a_w3)
    v_std, a_std = v.std(axis=1), a.std(axis=1)
    seg = _norm(p[:, 1:, :] - p[:, :-1, :])
    path_eff = _norm(p[:, -1, :] - p[:, 0, :]) / (seg.sum(1) + 1e-9)
    p0 = p[:, -1, :]
    distance_r = _norm(p0)
    radial_v = (p0 * v_last).sum(1) / (distance_r + 1e-9)
    v_n = v / (_norm(v)[..., None] + 1e-9)
    cos_consec = (v_n[:, 1:, :] * v_n[:, :-1, :]).sum(-1).clip(-1, 1)
    turn = np.arccos(cos_consec)
    turn_mean, cos_turn_last = turn.mean(1), cos_consec[:, -1]
    va_dot = (v_last * a_last).sum(1) / (speed_last * acc_norm_last + 1e-9)
    jerk_last = (a[:, -1, :] - a[:, -2, :]) / DT
    a_tang_last = (v_last * a_last).sum(1) / (speed_last + 1e-9)
    a_cent_last = _norm(np.cross(v_last, a_last)) / (speed_last + 1e-9)
    speed_seq = _norm(v)
    speed_diff_half = speed_seq[:, 5:].mean(1) - speed_seq[:, :5].mean(1)
    turn_mean_half_diff = turn[:, 4:].mean(1) - turn[:, :4].mean(1)
    a_N = (a_last * eN).sum(1); a_B = (a_last * eB).sum(1)
    j_L = (jerk_last * eL).sum(1); j_N = (jerk_last * eN).sum(1); j_B = (jerk_last * eB).sum(1)
    aw_L = (a_w3 * eL).sum(1); aw_N = (a_w3 * eN).sum(1)
    X = np.column_stack([
        speed_last, acc_norm_last, acc_norm_w3,
        v_std[:,0], v_std[:,1], v_std[:,2], a_std[:,0], a_std[:,1], a_std[:,2],
        path_eff, distance_r, radial_v, turn_mean, cos_turn_last, va_dot,
        a_tang_last, a_cent_last, speed_diff_half, turn_mean_half_diff,
        a_N, a_B, j_L, j_N, j_B, aw_L, aw_N,
    ]).astype(np.float32)
    return X


X_tr_v15 = build_v3_15(traj_train, eL_tr, eN_tr, eB_tr)
X_ts_v15 = build_v3_15(traj_test,  eL_ts, eN_ts, eB_ts)
X_tr_v26 = build_v3_26(traj_train, eL_tr, eN_tr, eB_tr)
X_ts_v26 = build_v3_26(traj_test,  eL_ts, eN_ts, eB_ts)
assert X_tr_v15.shape == (10000, 15) and X_ts_v15.shape == (10000, 15)
assert X_tr_v26.shape == (10000, 26) and X_ts_v26.shape == (10000, 26)
print(f'V3 15 feat: tr {X_tr_v15.shape}, ts {X_ts_v15.shape}')
print(f'V3 26 feat: tr {X_tr_v26.shape}, ts {X_ts_v26.shape}')


## 셀 5 — K15 + GRU cache 로드 + HR sanity + GRU form auto-detect

K15 (EXP #29): `topk_e29_{oof,test}_K15_seed*.npy` 3-seed avg residual → base + residual → HR.
GRU (EXP #25): `gru_e25_fren_{oof,test}_seed*.npy` — Cart vs Frenet form 자동 판별 (#31 셀 5.5 framework).
sanity 미달 시 lever skip 아니라 **lever 폐기** (GRU 필수).


In [ ]:
def load_seed_avg(paths):
    return np.mean([np.load(p) for p in paths], axis=0).astype(np.float32)


# K15
oof_K15 = load_seed_avg(K15_OOF_PATHS)
test_K15 = load_seed_avg(K15_TEST_PATHS)
assert oof_K15.shape == (10000, 3) and test_K15.shape == (10000, 3)

pred_K15_tr = base_train + oof_K15
pred_K15_ts = base_test + test_K15
err_K15 = np.linalg.norm(y_train - pred_K15_tr, axis=1)
hit_K15 = err_K15 < HIT_RADIUS
HR_K15 = hr_triple(hit_K15, minority_mask_tr)
print(f'K15 HR: overall={HR_K15["overall"]:.4f}  major={HR_K15["major"]:.4f}  minor={HR_K15["minor"]:.4f}')

# K15 sanity vs oracle_ceiling_l15.py
for k in ['overall', 'major', 'minor']:
    diff = abs(HR_K15[k] - K15_REF_HR[k])
    assert diff < 0.0005, f'K15 {k} drift: ref {K15_REF_HR[k]:.4f} vs got {HR_K15[k]:.4f} (Δ {diff:.4f})'
print('K15 sanity OK')

# GRU — Cart vs Frenet form auto-detect
oof_GRU = load_seed_avg(GRU_OOF_PATHS)
test_GRU = load_seed_avg(GRU_TEST_PATHS)
assert oof_GRU.shape == test_GRU.shape == (10000, 3)

pred_cart = base_train + oof_GRU
err_cart = np.linalg.norm(y_train - pred_cart, axis=1)
HR_cart_overall = float((err_cart < HIT_RADIUS).mean())

oof_GRU_inv = (oof_GRU[:, 0:1] * eL_tr + oof_GRU[:, 1:2] * eN_tr + oof_GRU[:, 2:3] * eB_tr)
pred_fren = base_train + oof_GRU_inv
err_fren = np.linalg.norm(y_train - pred_fren, axis=1)
HR_fren_overall = float((err_fren < HIT_RADIUS).mean())

print(f'\nGRU cache 형태 판별:')
print(f'  Cart 가정    HR overall = {HR_cart_overall:.4f}  (|Δ vs {GRU_REF_OVERALL}| = {abs(HR_cart_overall - GRU_REF_OVERALL):.4f})')
print(f'  Frenet 가정  HR overall = {HR_fren_overall:.4f}  (|Δ vs {GRU_REF_OVERALL}| = {abs(HR_fren_overall - GRU_REF_OVERALL):.4f})')

if abs(HR_cart_overall - GRU_REF_OVERALL) < 0.005:
    pred_GRU_tr = pred_cart
    pred_GRU_ts = base_test + test_GRU
    gru_format = 'cart'
elif abs(HR_fren_overall - GRU_REF_OVERALL) < 0.005:
    pred_GRU_tr = pred_fren
    test_GRU_inv = (test_GRU[:, 0:1] * eL_ts + test_GRU[:, 1:2] * eN_ts + test_GRU[:, 2:3] * eB_ts)
    pred_GRU_ts = base_test + test_GRU_inv
    gru_format = 'frenet'
else:
    raise RuntimeError(f'GRU cache 형태 불명 — Cart/Frenet 둘 다 HR sanity fail. EXP #32 진행 불가')

err_GRU = np.linalg.norm(y_train - pred_GRU_tr, axis=1)
hit_GRU = err_GRU < HIT_RADIUS
HR_GRU = hr_triple(hit_GRU, minority_mask_tr)
print(f'\nGRU 채택 형태: {gru_format}')
print(f'GRU HR: overall={HR_GRU["overall"]:.4f}  major={HR_GRU["major"]:.4f}  minor={HR_GRU["minor"]:.4f}')

# GRU minor sanity vs #25 보고
diff_minor = abs(HR_GRU['minor'] - GRU_REF_MINOR)
if diff_minor > 0.005:
    print(f'⚠ GRU minor drift: ref {GRU_REF_MINOR:.4f} vs got {HR_GRU["minor"]:.4f} (Δ {diff_minor:.4f}) — cache 약간 불일치')
else:
    print(f'GRU minor sanity OK ({HR_GRU["minor"]:.4f} ≈ #25 보고 {GRU_REF_MINOR})')

# K15/GRU 2-way oracle ceiling
choose_GRU = err_GRU < err_K15
pred_oracle_KG = np.where(choose_GRU[:, None], pred_GRU_tr, pred_K15_tr)
err_oracle_KG = np.linalg.norm(y_train - pred_oracle_KG, axis=1)
hit_oracle_KG = err_oracle_KG < HIT_RADIUS
HR_oracle_KG = hr_triple(hit_oracle_KG, minority_mask_tr)
print(f'\nK15/GRU 2-way oracle ceiling: overall={HR_oracle_KG["overall"]:.4f}  major={HR_oracle_KG["major"]:.4f}  minor={HR_oracle_KG["minor"]:.4f}')
print(f'  Δ vs K15: overall +{HR_oracle_KG["overall"]-K15_REF_HR["overall"]:.4f}  minor +{HR_oracle_KG["minor"]-K15_REF_HR["minor"]:.4f}')


## 셀 6 — Phase 0: selector target + agreement + recovery_mask

`y_sel_kg = (err_GRU < err_K15).astype(int)` — 1 = GRU wins (use GRU), 0 = K15 wins.

**recovery_mask = K15_only | GRU_only** = selector 결정이 HR 영향 주는 sample (~4.3%).
이 영역에서 직접 측정한 AUC가 primary sanity (overall AUC는 비-recovery 영역으로 dominated).


In [ ]:
y_sel_kg = (err_GRU < err_K15).astype(np.int8)
n_GRUw = int(y_sel_kg.sum())
n_K15w = int((y_sel_kg == 0).sum())
maj_baseline_kg = max(n_GRUw, n_K15w) / len(y_sel_kg)
print(f'GRU wins: {n_GRUw} ({n_GRUw/len(y_sel_kg):.4f}) | K15 wins: {n_K15w} ({n_K15w/len(y_sel_kg):.4f})')
print(f'Majority class baseline accuracy: {maj_baseline_kg:.4f}')

# Agreement breakdown K15 vs GRU
both_kg = hit_K15 & hit_GRU
K15_only_kg = hit_K15 & ~hit_GRU
GRU_only_kg = ~hit_K15 & hit_GRU
neither_kg = ~hit_K15 & ~hit_GRU
print(f'\nAgreement K15/GRU: both {int(both_kg.sum())} | K15_only {int(K15_only_kg.sum())} | GRU_only {int(GRU_only_kg.sum())} | neither {int(neither_kg.sum())}')

# Recovery mask — selector 결정이 hit/miss 결정하는 영역
recovery_mask = K15_only_kg | GRU_only_kg
print(f'Recovery mask: {int(recovery_mask.sum())} sample ({recovery_mask.mean():.4f})')

# Recovery 영역 내 minority/majority 분포
rec_minor_count = int((recovery_mask & minority_mask_tr).sum())
rec_major_count = int((recovery_mask & ~minority_mask_tr).sum())
print(f'  recovery 내 minority: {rec_minor_count} / majority: {rec_major_count}')

# Recovery-focused classifier overall acc 상한 (직관 검증)
acc_max_rec_focused = (recovery_mask.sum() * 1.0 + (~recovery_mask).sum() * 0.5) / 10000
print(f'\nRecovery-focused classifier overall acc 상한 (recovery 100% + 비-recovery 50% 가정): {acc_max_rec_focused:.4f}')
print(f'→ overall acc 0.55 임계는 recovery-focused classifier를 부당하게 차단 가능')
print(f'→ recovery-area AUC > {AUC_RECOVERY_THR} primary sanity 적용')


## 셀 7 — Phase 1: V3 15 LGBM classifier (3 seed × 5 fold = 15 model)

V3 15 feat → K15 vs GRU binary classifier. StratifiedKFold (stratify=minority_mask) — K15/GRU cache OOF와 fold-aligned 필수.

가드: **recovery-area AUC > 0.55** (primary sanity). overall AUC > 0.50 (sanity). 미달 시 Phase 2a 자동 trigger.


In [ ]:
print('=== Phase 1: V3 15 LGBM classifier 학습 ===')
_t_p1 = time.time()
phase1_result = run_classifier_seeds(
    X_tr=X_tr_v15, y_target=y_sel_kg, X_ts=X_ts_v15,
    stratify=minority_mask_tr.astype(int),
)
print(f'Phase 1 학습 시간: {time.time()-_t_p1:.0f}s')

phase1_metrics = compute_classifier_metrics(
    phase1_result['oof_prob_mean'], y_sel_kg,
    recovery_mask=recovery_mask, per_seed=phase1_result['per_seed'],
)
for m in phase1_result['per_seed']:
    print(f'  seed={m["seed"]}: acc={m["accuracy"]:.4f}  AUC={m["auc"]:.4f}  fold_std={m["fold_std"]:.4f}  ({m["time_sec"]:.0f}s)')

print(f'\n3-seed Phase 1 metrics:')
print(f'  overall acc       = {phase1_metrics["overall_acc"]:.4f}  (informational only, baseline {maj_baseline_kg:.4f})')
print(f'  overall AUC       = {phase1_metrics["overall_auc"]:.4f}  → {"PASS" if phase1_metrics["overall_auc"] > AUC_OVERALL_SANITY else "FAIL"} (sanity > {AUC_OVERALL_SANITY})')
print(f'  recovery-area AUC = {phase1_metrics["recovery_auc"]:.4f}  → {"PASS O (primary)" if phase1_metrics["recovery_auc"] > AUC_RECOVERY_THR else "FAIL X (primary)"} (> {AUC_RECOVERY_THR})')
print(f'  fold std mean     = {phase1_metrics["fold_std_mean"]:.4f}  → {"PASS" if phase1_metrics["fold_std_mean"] < FOLD_STD_MAX else "WARN"} (< {FOLD_STD_MAX})')

# Save Phase 1 classifier (per-seed OOF/test + mean)
for i, seed in enumerate(SEEDS):
    np.save(f'results/split_e32_clf_oof_seed{seed}.npy', phase1_result['oof_probs_per_seed'][i])
    np.save(f'results/split_e32_clf_test_seed{seed}.npy', phase1_result['test_probs_per_seed'][i])
np.save('results/split_e32_phase1_oof_mean.npy', phase1_result['oof_prob_mean'])
np.save('results/split_e32_phase1_test_mean.npy', phase1_result['test_prob_mean'])

phase1_pass = phase1_metrics['recovery_auc'] > AUC_RECOVERY_THR
print(f'\n→ Phase 1 primary sanity (recovery-area AUC > {AUC_RECOVERY_THR}): {"PASS — Phase 2a skip" if phase1_pass else "FAIL — Phase 2a 자동 trigger"}')


## 셀 8 — Phase 2a (조건부 자동 trigger): expanded feature ablation

Phase 1 recovery-area AUC < 0.55 시 자동 진입. 3 lever 순차 학습 (각 ~5분, 총 ~15분):
1. **V3 26** (26 feat) — simplicity 제거 11 feat 복원
2. **Residual-derived** (8 feat) — K15 OOF residual 3축+norm + GRU OOF residual 3축+norm
3. **Hybrid** (23 feat) — V3 15 + Residual-derived

1 lever라도 recovery-area AUC > 0.55 통과 시 해당 lever 채택. 모두 미달이어도 Phase 3 진행 (HR 4th guard primary).

**경고**: Residual-derived는 K15·GRU OOF residual 입력 — fold partition 동일 보장 필수 (StratifiedKFold seed/stratify #31 동일).


In [ ]:
phase2a_results = {}
phase2a_run = not phase1_pass  # 자동 trigger

if phase2a_run:
    print('=== Phase 2a: V3 26 / Residual-derived / Hybrid 3 lever 순차 학습 ===\n')

    # Residual-derived feature 구성 (K15 OOF residual + GRU OOF residual)
    # oof_K15: (10000, 3) = label - base - oof_K15 인지 base + oof_K15 = pred 인지 일관성 — 셀 5에서 base + oof_K15 = pred 이므로 residual_K15_oof = y - pred
    resid_K15 = (y_train - pred_K15_tr).astype(np.float32)
    resid_GRU = (y_train - pred_GRU_tr).astype(np.float32)
    norm_K15 = _norm(resid_K15)[:, None]
    norm_GRU = _norm(resid_GRU)[:, None]
    X_tr_resid = np.column_stack([resid_K15, norm_K15, resid_GRU, norm_GRU]).astype(np.float32)
    assert X_tr_resid.shape == (10000, 8)

    # Test residual은 train과 동일 방식 — 단 test residual은 GT 없으므로 사용 불가
    # 대신 test에서는 K15/GRU prediction의 magnitude 등을 proxy로 — Phase 3에서 test 예측 위해
    # NOTE: Residual-derived feature는 train OOF residual 의존 → test 적용 시 leakage 위험
    # 안전 strategy: Residual-derived는 train OOF 전용, test prob은 V3 15/V3 26 classifier 사용
    # 본 lever 채택 시에도 test classifier는 V3 15 또는 V3 26으로 대체
    print(f'Residual-derived feature: tr {X_tr_resid.shape} (test 미사용 — leakage 회피)')

    X_tr_hybrid = np.column_stack([X_tr_v15, X_tr_resid]).astype(np.float32)
    assert X_tr_hybrid.shape == (10000, 23)

    p2a_lever_list = [
        ('v3_26',     X_tr_v26,    X_ts_v26,    'V3 26 feat'),
        ('resid',     X_tr_resid,  None,        'Residual-derived 8 feat (test 미사용)'),
        ('hybrid',    X_tr_hybrid, None,        'Hybrid V3 15 + Residual 23 feat (test 미사용)'),
    ]

    for lev_name, X_tr_lev, X_ts_lev, desc in p2a_lever_list:
        print(f'--- Phase 2a {lev_name}: {desc} ---')
        _t = time.time()
        res = run_classifier_seeds(X_tr=X_tr_lev, y_target=y_sel_kg, X_ts=X_ts_lev,
                                    stratify=minority_mask_tr.astype(int))
        mets = compute_classifier_metrics(res['oof_prob_mean'], y_sel_kg,
                                           recovery_mask=recovery_mask, per_seed=res['per_seed'])
        rec_pass = mets['recovery_auc'] > AUC_RECOVERY_THR
        phase2a_results[lev_name] = dict(result=res, metrics=mets, pass_recovery=rec_pass,
                                          has_test=X_ts_lev is not None,
                                          time_sec=time.time() - _t)
        print(f'  overall acc={mets["overall_acc"]:.4f}  AUC={mets["overall_auc"]:.4f}  '
              f'rec-AUC={mets["recovery_auc"]:.4f}  fold_std={mets["fold_std_mean"]:.4f}  '
              f'→ recovery {"PASS O" if rec_pass else "FAIL X"} ({time.time()-_t:.0f}s)\n')
else:
    print('Phase 1 통과 → Phase 2a skip')


## 셀 9 — Phase 2b (always run): minority-restricted classifier

minority subset (1575 sample)에서만 학습. StratifiedKFold stratify=y_sel_kg[minority_mask] (subset 내 class balance).
V3 15 feature 사용 (Phase 1과 동일 feature space에서 minority 농축 효과 측정).

가드: minority subset recovery-area AUC > 0.55 (minority 내 K15_only/GRU_only sample만으로 측정).


In [ ]:
print('=== Phase 2b: minority-restricted classifier (V3 15 on minority subset n=1575) ===')

# Minority subset index
minor_idx = np.where(minority_mask_tr)[0]
y_sel_kg_minor = y_sel_kg[minor_idx]
X_tr_v15_minor = X_tr_v15[minor_idx]

# Minority 내 class 분포 + recovery mask
n_GRUw_minor = int(y_sel_kg_minor.sum())
n_K15w_minor = int((y_sel_kg_minor == 0).sum())
recovery_mask_minor_subset = recovery_mask[minor_idx]
print(f'Minority subset: {len(minor_idx)} sample, GRU wins {n_GRUw_minor} / K15 wins {n_K15w_minor}')
print(f'Minority recovery: {recovery_mask_minor_subset.sum()} sample (minority 내 K15_only ∪ GRU_only)')

# Subset 내 stratify = y_sel_kg_minor (test는 별도 학습, OOF는 fold-CV)
_t_p2b = time.time()
phase2b_result_sub = run_classifier_seeds(
    X_tr=X_tr_v15_minor, y_target=y_sel_kg_minor, X_ts=None,
    stratify=y_sel_kg_minor.astype(int),
)
print(f'Phase 2b 학습 시간: {time.time()-_t_p2b:.0f}s')

# Map subset oof back to 10000-length array (NaN outside minority)
phase2b_oof_full = np.full(10000, np.nan, dtype=np.float32)
phase2b_oof_full[minor_idx] = phase2b_result_sub['oof_prob_mean']

# Phase 2b test classifier: 별도 학습 (전체 minority subset로 학습 → 전체 10000 test에 적용)
print('  Phase 2b test classifier: 전체 minority subset 학습 → test 전체 적용')
phase2b_test = np.zeros(10000, dtype=np.float32)
for seed in SEEDS:
    m = lgb.LGBMClassifier(**{**clf_params, 'random_state': seed})
    m.fit(X_tr_v15_minor, y_sel_kg_minor)
    phase2b_test += m.predict_proba(X_ts_v15)[:, 1].astype(np.float32) / len(SEEDS)

# Phase 2b metrics (minority subset only)
phase2b_metrics = compute_classifier_metrics(
    phase2b_result_sub['oof_prob_mean'], y_sel_kg_minor,
    recovery_mask=recovery_mask_minor_subset, per_seed=phase2b_result_sub['per_seed'],
)
print(f'\n3-seed Phase 2b metrics (minority subset):')
print(f'  overall acc       = {phase2b_metrics["overall_acc"]:.4f}  (subset baseline)')
print(f'  overall AUC       = {phase2b_metrics["overall_auc"]:.4f}  → {"PASS" if phase2b_metrics["overall_auc"] > AUC_OVERALL_SANITY else "FAIL"} (> {AUC_OVERALL_SANITY})')
print(f'  recovery-area AUC = {phase2b_metrics["recovery_auc"]:.4f}  → {"PASS O" if phase2b_metrics["recovery_auc"] > AUC_RECOVERY_THR else "FAIL X"} (minority 내 recovery)')
print(f'  fold std mean     = {phase2b_metrics["fold_std_mean"]:.4f}')

phase2b_pass = phase2b_metrics['recovery_auc'] > AUC_RECOVERY_THR
print(f'\n→ Phase 2b recovery-area AUC > {AUC_RECOVERY_THR}: {"PASS" if phase2b_pass else "FAIL"}')


## 셀 10 — Phase 3: 6 mode HR + 3-gate + 4th guard + floor

**Primary classifier 선정**: Phase 1/2a 중 recovery-area AUC max. Phase 1 pass면 V3 15, 아니면 Phase 2a 통과 lever 중 max (test prediction 가용한 V3 26 우선), 모두 미달이면 V3 15 (fallback).

**6 mode**:
1. Hard p>0.5 → GRU else K15
2. Soft w·GRU + (1−w)·K15
3. Threshold p>0.7 → GRU else K15
4. Threshold p>0.8 → GRU else K15
5. Minor-restricted Hard: minority_mask & (Phase 2b clf p>0.5) → GRU else K15
6. Minor→GRU rule (학습 0): minority_mask → GRU else K15 (floor)

**Gate**: G1 multi-comparison +0.0034 (√6) / G2 major −0.002 / G3 minor −0.005 / G4 per-mode +0.0020 / Floor +0.0012.


In [ ]:
# Primary classifier 선정 — recovery-area AUC max + test prediction 가용
candidates_primary = [('phase1_v15', phase1_result, phase1_metrics, True)]
if phase2a_run:
    for name, info in phase2a_results.items():
        candidates_primary.append((f'phase2a_{name}', info['result'], info['metrics'], info['has_test']))

# test 가용한 lever 중 recovery AUC max 우선
candidates_with_test = [c for c in candidates_primary if c[3]]
if candidates_with_test:
    candidates_with_test.sort(key=lambda c: -c[2]['recovery_auc'])
    primary_name, primary_res, primary_mets, _ = candidates_with_test[0]
    print(f'Primary classifier 선정: {primary_name} (recovery-AUC {primary_mets["recovery_auc"]:.4f})')
else:
    primary_name = 'phase1_v15'
    primary_res = phase1_result
    primary_mets = phase1_metrics
    print(f'Primary classifier 선정 (fallback): {primary_name}')

oof_prob_primary = primary_res['oof_prob_mean']  # 1 = GRU confidence
test_prob_primary = primary_res['test_prob_mean']

# ── Mode 정의 ──
# Mode 1: Hard p>0.5
use_GRU_hard = oof_prob_primary > 0.5
pred_hard = np.where(use_GRU_hard[:, None], pred_GRU_tr, pred_K15_tr)

# Mode 2: Soft
w_soft = oof_prob_primary[:, None]
pred_soft = w_soft * pred_GRU_tr + (1 - w_soft) * pred_K15_tr

# Mode 3: p>0.7
use_GRU_p07 = oof_prob_primary > 0.7
pred_p07 = np.where(use_GRU_p07[:, None], pred_GRU_tr, pred_K15_tr)

# Mode 4: p>0.8
use_GRU_p08 = oof_prob_primary > 0.8
pred_p08 = np.where(use_GRU_p08[:, None], pred_GRU_tr, pred_K15_tr)

# Mode 5: Minor-restricted Hard (Phase 2b classifier on minority subset)
use_GRU_mr = np.zeros(10000, dtype=bool)
phase2b_oof_safe = np.where(np.isnan(phase2b_oof_full), 0.0, phase2b_oof_full)
use_GRU_mr[minority_mask_tr] = phase2b_oof_safe[minority_mask_tr] > 0.5
pred_mrhard = np.where(use_GRU_mr[:, None], pred_GRU_tr, pred_K15_tr)

# Mode 6: Minor→GRU rule (학습 0 floor)
pred_minorgru = np.where(minority_mask_tr[:, None], pred_GRU_tr, pred_K15_tr)

# ── HR 측정 ──
def _eval_mode(pred):
    err = np.linalg.norm(y_train - pred, axis=1)
    hit = err < HIT_RADIUS
    return hr_triple(hit, minority_mask_tr), hit

HR_hard, hit_hard = _eval_mode(pred_hard)
HR_soft, hit_soft = _eval_mode(pred_soft)
HR_p07, hit_p07 = _eval_mode(pred_p07)
HR_p08, hit_p08 = _eval_mode(pred_p08)
HR_mrhard, hit_mrhard = _eval_mode(pred_mrhard)
HR_minorgru, hit_minorgru = _eval_mode(pred_minorgru)

print(f'=== HR summary (vs K15 ref {K15_REF_HR["overall"]:.4f}) ===')
print(f'{"row":<14} {"overall":>9}  {"major":>9}  {"minor":>9}  {"Δo":>8}  {"Δm":>8}')
print('-' * 76)
for name, HR in [
    ('K15 ref',         K15_REF_HR),
    ('GRU ref',         HR_GRU),
    ('Oracle K15/GRU',  HR_oracle_KG),
]:
    do = HR['overall'] - K15_REF_HR['overall']
    dm = HR['minor'] - K15_REF_HR['minor']
    do_s = '—' if abs(do) < 1e-9 else f'{do:+.4f}'
    dm_s = '—' if abs(dm) < 1e-9 else f'{dm:+.4f}'
    print(f'{name:<14} {HR["overall"]:.4f}     {HR["major"]:.4f}     {HR["minor"]:.4f}     {do_s:>8}  {dm_s:>8}')
print('-' * 76)
mode_HRs = [
    ('Hard p>0.5',   HR_hard),
    ('Soft',         HR_soft),
    ('Thr p>0.7',    HR_p07),
    ('Thr p>0.8',    HR_p08),
    ('MR-Hard',      HR_mrhard),
    ('MinorGRU',     HR_minorgru),
]
for name, HR in mode_HRs:
    do = HR['overall'] - K15_REF_HR['overall']
    dm = HR['minor'] - K15_REF_HR['minor']
    print(f'{name:<14} {HR["overall"]:.4f}     {HR["major"]:.4f}     {HR["minor"]:.4f}     {do:+.4f}   {dm:+.4f}')

# ── Gate 평가 ──
def evaluate_gate(name, HR, floor_do=FLOOR_DO):
    d_o = HR['overall'] - K15_REF_HR['overall']
    d_M = HR['major'] - K15_REF_HR['major']
    d_m = HR['minor'] - K15_REF_HR['minor']
    g1 = d_o > G1_THR
    g2 = d_M > -0.002
    g3 = d_m > -0.005
    g4 = d_o > G4_THR
    floor_pass = d_o > floor_do
    lb_ok = bool(g4 and g2 and g3 and floor_pass)
    return dict(name=name, d_overall=float(d_o), d_major=float(d_M), d_minor=float(d_m),
                G1=bool(g1), G2=bool(g2), G3=bool(g3), G4=bool(g4), Floor=bool(floor_pass),
                lb_eligible=lb_ok)

gates = []
mode_data = [
    ('Hard',     'hard',     HR_hard,     pred_hard,     'Hard p>0.5 → GRU else K15'),
    ('Soft',     'soft',     HR_soft,     pred_soft,     'w·GRU + (1−w)·K15'),
    ('p07',      'p07',      HR_p07,      pred_p07,      'p>0.7 → GRU else K15'),
    ('p08',      'p08',      HR_p08,      pred_p08,      'p>0.8 → GRU else K15'),
    ('MRHard',   'mrhard',   HR_mrhard,   pred_mrhard,   'minority_mask & (P2b p>0.5) → GRU else K15'),
    ('MinorGRU', 'minorgru', HR_minorgru, pred_minorgru, 'minority_mask → GRU else K15 (floor)'),
]
# Floor mode 자체는 floor 가드 무시 (rule baseline이라 자기 자신과 비교 의미 없음 — 4th guard만 적용)
for full_name, short_name, HR, _, _ in mode_data:
    floor = 0.0 if short_name == 'minorgru' else FLOOR_DO
    gates.append((short_name, evaluate_gate(full_name, HR, floor_do=floor)))

print(f'\n=== 3-gate + 4th guard + Floor (vs K15, G1=√6·{COMBINED_STD:.4f}={G1_THR:.4f}, G4=√2·{COMBINED_STD:.4f}={G4_THR:.4f}, Floor=+{FLOOR_DO:.4f}) ===')
for short_name, g in gates:
    print(f'  {g["name"]:<10} Δo={g["d_overall"]:+.4f}  ΔM={g["d_major"]:+.4f}  Δm={g["d_minor"]:+.4f}  '
          f'G1={"O" if g["G1"] else "X"} G2={"O" if g["G2"] else "X"} G3={"O" if g["G3"] else "X"} G4={"O" if g["G4"] else "X"} F={"O" if g["Floor"] else "X"}  '
          f'LB={"O" if g["lb_eligible"] else "X"}')

# Recall analysis
def _recall_kg(use_GRU_arr, name):
    GRU_only_recovered = int((use_GRU_arr & GRU_only_kg).sum())
    K15_only_lost = int((use_GRU_arr & K15_only_kg).sum())
    n_GRU = int(use_GRU_arr.sum())
    net = GRU_only_recovered - K15_only_lost
    return dict(name=name, n_GRU_selected=n_GRU,
                GRU_only_recovered=GRU_only_recovered, K15_only_lost=K15_only_lost,
                net_gain_hit=net, oracle_max_gain=int(GRU_only_kg.sum()))

print(f'\n=== Recall analysis (K15/GRU agreement, GRU_only {int(GRU_only_kg.sum())} / K15_only {int(K15_only_kg.sum())}) ===')
recall_hard     = _recall_kg(use_GRU_hard,                 'Hard p>0.5')
recall_p07      = _recall_kg(use_GRU_p07,                  'Thr p>0.7')
recall_p08      = _recall_kg(use_GRU_p08,                  'Thr p>0.8')
recall_mrhard   = _recall_kg(use_GRU_mr,                   'MR-Hard')
recall_minorgru = _recall_kg(minority_mask_tr.astype(bool),'MinorGRU rule')
for r in [recall_hard, recall_p07, recall_p08, recall_mrhard, recall_minorgru]:
    print(f'  {r["name"]:<14} GRU 선택 {r["n_GRU_selected"]:5d}  GRU_only 회수 {r["GRU_only_recovered"]:3d}/{r["oracle_max_gain"]:3d}  K15_only 손실 {r["K15_only_lost"]:3d}  net {r["net_gain_hit"]:+d}')


## 셀 11 — Submission 생성 (6 mode 무조건) + Drive 복사 + files.download

**plan §4.17 + memory feedback-submission-unconditional**: guard fail이어도 6 mode 전부 무조건 생성. LB 제출 결정은 별도 (`lb_eligible_modes`로 메타에 기록).


In [ ]:
def make_submission(pred_test_abs, name):
    assert pred_test_abs.shape == (10000, 3)
    assert np.isfinite(pred_test_abs).all()
    df = pd.DataFrame({
        'id': test_ids,
        'x': pred_test_abs[:, 0],
        'y': pred_test_abs[:, 1],
        'z': pred_test_abs[:, 2],
    })
    path = f'submission_split_e32_{name}_ms3.csv'
    df.to_csv(path, index=False)
    print(f'  {path}: x[{pred_test_abs[:,0].min():.2f},{pred_test_abs[:,0].max():.2f}] '
          f'y[{pred_test_abs[:,1].min():.2f},{pred_test_abs[:,1].max():.2f}] '
          f'z[{pred_test_abs[:,2].min():.2f},{pred_test_abs[:,2].max():.2f}]')
    return path


# Test prediction 계산 (6 mode 전부 무조건)
print('=== Test prediction + submission 생성 (6 mode 무조건) ===')

# Modes 1-4: primary classifier test prob
use_GRU_test_hard = test_prob_primary > 0.5
pred_test_hard = np.where(use_GRU_test_hard[:, None], pred_GRU_ts, pred_K15_ts)

w_test_soft = test_prob_primary[:, None]
pred_test_soft = w_test_soft * pred_GRU_ts + (1 - w_test_soft) * pred_K15_ts

use_GRU_test_p07 = test_prob_primary > 0.7
pred_test_p07 = np.where(use_GRU_test_p07[:, None], pred_GRU_ts, pred_K15_ts)

use_GRU_test_p08 = test_prob_primary > 0.8
pred_test_p08 = np.where(use_GRU_test_p08[:, None], pred_GRU_ts, pred_K15_ts)

# Mode 5: Phase 2b classifier on minority_mask_ts only
use_GRU_test_mr = np.zeros(10000, dtype=bool)
use_GRU_test_mr[minority_mask_ts] = phase2b_test[minority_mask_ts] > 0.5
pred_test_mrhard = np.where(use_GRU_test_mr[:, None], pred_GRU_ts, pred_K15_ts)

# Mode 6: Minor→GRU rule (no classifier)
pred_test_minorgru = np.where(minority_mask_ts[:, None], pred_GRU_ts, pred_K15_ts)

sub_paths = {}
test_preds_by_mode = [
    ('hard',     pred_test_hard,     HR_hard,     gates[0][1]),
    ('soft',     pred_test_soft,     HR_soft,     gates[1][1]),
    ('p07',      pred_test_p07,      HR_p07,      gates[2][1]),
    ('p08',      pred_test_p08,      HR_p08,      gates[3][1]),
    ('mrhard',   pred_test_mrhard,   HR_mrhard,   gates[4][1]),
    ('minorgru', pred_test_minorgru, HR_minorgru, gates[5][1]),
]
print(f'\nTest minority count: {int(minority_mask_ts.sum())}/{len(minority_mask_ts)} ({minority_mask_ts.mean():.4f})')
print(f'Mode 5 (MR-Hard) test GRU 선택: {int(use_GRU_test_mr.sum())} sample (minority_mask_ts ∩ phase2b_test > 0.5)')
print(f'Mode 6 (MinorGRU) test GRU 선택: {int(minority_mask_ts.sum())} sample (minority_mask_ts 전체)\n')

for mode_name, pred_test, HR_oof, gate in test_preds_by_mode:
    path = make_submission(pred_test, mode_name)
    sub_paths[mode_name] = path

# LB 제출 우선순위 (정보만, 실제 LB 제출은 사용자 결정)
lb_eligible = [(short, g) for short, g in gates if g['lb_eligible']]
lb_eligible.sort(key=lambda x: -x[1]['d_overall'])

print(f'\n=== LB 제출 정책 (정보) ===')
if not lb_eligible:
    print(f'  4th guard +{G4_THR:.4f} AND Floor +{FLOOR_DO:.4f} 둘 다 통과 mode: 0개')
    print(f'  → LB 슬롯 사용 권장 안 함 (6 submission은 정량 자산 보존)')
    chosen_lb = None
else:
    chosen_lb = lb_eligible[0][0]
    print(f'  LB eligible mode ({len(lb_eligible)}개):')
    for short_name, g in lb_eligible:
        print(f'    {g["name"]:<10} Δo={g["d_overall"]:+.4f}  →  submission_split_e32_{short_name}_ms3.csv')
    print(f'  → best mode (OOF Δo max): **{chosen_lb}** (LB 슬롯 1개 권장)')


## 셀 12 — Meta 저장 + Drive 복사 + files.download (6 mode 무조건)

In [ ]:
def safe(x):
    if isinstance(x, np.floating): return float(x)
    if isinstance(x, np.integer): return int(x)
    if isinstance(x, np.bool_): return bool(x)
    if isinstance(x, np.ndarray): return x.tolist()
    return x


meta = {
    'protocol': 'EXP #32 L16 K15/GRU Selector (heterogeneous backbone per-sample switch)',
    'seeds': SEEDS, 'n_folds': N_FOLDS,
    'minority_threshold': MINORITY_THRESHOLD,
    'thresholds': {
        'auc_recovery_thr': AUC_RECOVERY_THR,
        'auc_overall_sanity': AUC_OVERALL_SANITY,
        'fold_std_max': FOLD_STD_MAX,
        'g1_thr_multi_comparison': G1_THR,
        'g4_thr_per_mode': G4_THR,
        'floor_do': FLOOR_DO,
        'combined_std': COMBINED_STD,
    },
    'k15_ref_HR': K15_REF_HR,
    'gru_ref_overall': GRU_REF_OVERALL,
    'gru_ref_minor': GRU_REF_MINOR,
    'gru_format': gru_format,
    'gru_HR': HR_GRU,
    'oracle_HR_K15GRU': HR_oracle_KG,
    'ceiling_gain_K15GRU': CEILING_GAIN_KG,
    'agreement_K15_GRU': dict(both=int(both_kg.sum()), K15_only=int(K15_only_kg.sum()),
                                GRU_only=int(GRU_only_kg.sum()), neither=int(neither_kg.sum())),
    'recovery_mask_count': int(recovery_mask.sum()),
    'y_sel_kg_distribution': dict(GRU_wins=n_GRUw, K15_wins=n_K15w, maj_baseline=maj_baseline_kg),
    'phase1': {
        'feature': 'V3 15',
        'per_seed': phase1_result['per_seed'],
        'metrics': phase1_metrics,
        'pass_recovery_auc': bool(phase1_pass),
    },
    'phase2a_run': bool(phase2a_run),
    'phase2a_results': {
        name: {
            'per_seed': info['result']['per_seed'],
            'metrics': info['metrics'],
            'pass_recovery_auc': bool(info['pass_recovery']),
            'has_test_prediction': bool(info['has_test']),
            'time_sec': info['time_sec'],
        }
        for name, info in phase2a_results.items()
    } if phase2a_run else {},
    'phase2b': {
        'feature': 'V3 15 on minority subset',
        'subset_size': int(len(minor_idx)),
        'subset_y_distribution': dict(GRU_wins=n_GRUw_minor, K15_wins=n_K15w_minor),
        'per_seed': phase2b_result_sub['per_seed'],
        'metrics': phase2b_metrics,
        'pass_recovery_auc': bool(phase2b_pass),
    },
    'primary_classifier': primary_name,
    'modes': {
        'hard':     dict(HR=HR_hard,     gate=gates[0][1], recall=recall_hard),
        'soft':     dict(HR=HR_soft,     gate=gates[1][1]),
        'p07':      dict(HR=HR_p07,      gate=gates[2][1], recall=recall_p07),
        'p08':      dict(HR=HR_p08,      gate=gates[3][1], recall=recall_p08),
        'mrhard':   dict(HR=HR_mrhard,   gate=gates[4][1], recall=recall_mrhard),
        'minorgru': dict(HR=HR_minorgru, gate=gates[5][1], recall=recall_minorgru),
    },
    'lb_eligible_modes': [name for name, g in lb_eligible],
    'lb_chosen': chosen_lb,
    'submissions': sub_paths,
    'minority_train_count': int(minority_mask_tr.sum()),
    'minority_test_count': int(minority_mask_ts.sum()),
}
with open('results/split_e32_meta.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False, default=safe)
print('results/split_e32_meta.json 저장')

# Drive 복사 + files.download (6 mode 무조건)
try:
    from google.colab import drive, files
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    DRIVE_BASE = '/content/drive/MyDrive'
    results_drive = os.path.join(DRIVE_BASE, 'results')
    os.makedirs(results_drive, exist_ok=True)
    !cp -r results/* {results_drive}/
    if os.path.exists('hr_aware_split_e32.ipynb'):
        !cp hr_aware_split_e32.ipynb {DRIVE_BASE}/
    for p in sub_paths.values():
        !cp {p} {DRIVE_BASE}/
        files.download(p)
    print(f'Drive 복사 + 다운로드 완료 ({len(sub_paths)} submission 전부)')
except ImportError:
    print('Colab 아님 — Drive 복사 skip')
